# Cavendish Video Tracking
Tracks the moving mass in `periodo.avi`, annotates the video, and exports pixel and calibrated coordinates.

In [2]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.figsize'] = (8, 6)

In [3]:
def show_bgr(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.imshow(rgb)
    plt.axis('off')
    return rgb

def make_mask(image, polygon, show=False):
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.drawContours(mask, [polygon], -1, 255, -1)
    if show:
        plt.imshow(mask, cmap='gray'); plt.axis('off')
    return mask

def color_filter(image, lower, upper, in_bgr=False, show=False):
    space = image if in_bgr else cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    binary = cv2.inRange(space, lower, upper)
    if show:
        plt.imshow(binary, cmap='gray'); plt.axis('off')
    return binary

def find_centers(mask, min_area):
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    centers = []
    for contour in contours:
        if cv2.contourArea(contour) < min_area:
            continue
        m = cv2.moments(contour)
        centers.append((m['m10']/m['m00'], m['m01']/m['m00']))
    return centers

def perspective_coords(x_px, y_px, calibration_pixels, calibration_coords):
    matrix = cv2.getPerspectiveTransform(calibration_pixels, calibration_coords)
    homogeneous = np.array([[x_px], [y_px], [1]], dtype=np.float32)
    mapped = matrix @ homogeneous
    mapped /= mapped[2]
    return mapped[0, 0], mapped[1, 0]

def read_points_file(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                data.append((float(parts[0]), float(parts[1])))
    return data

In [4]:
# Paths and thresholds
video_path = Path('periodo.avi')
annotated_video_path = Path('periodo_track.avi')
pixels_output = Path('distanze_prova_periodo.txt')  # pixel centers from tracking
calibrated_output = Path('distanze_prova_periodo_calibrated.txt')

# Color thresholds (HSV) for the moving marker
lower_marker = np.array([140, 20, 100])
upper_marker = np.array([180, 100, 255])

# Regions of interest (pixel corners)
roi_moving = np.array([[500, 400], [500, 800], [1000, 800], [1000, 400]])
roi_fixed = np.array([[750, 500], [750, 700], [900, 700], [900, 500]])
roi_calibration = np.array([[150, 200], [150, 1000], [1900, 1000], [1900, 200]])

In [5]:
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise FileNotFoundError(f'Cannot open {video_path}')
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ret, frame0 = cap.read()
cap.release()
print(f'Frames: {n_frames}, shape: {frame0.shape}')
show_bgr(frame0)

FileNotFoundError: Cannot open periodo.avi

## Detect moving marker in one frame

In [ ]:
mask = make_mask(frame0, roi_moving)
masked = cv2.bitwise_and(frame0, frame0, mask=mask)
binary = color_filter(masked, lower_marker, upper_marker)
centers = find_centers(binary, min_area=150)
print('Centers found:', centers)
show_bgr(frame0)
for x_c, y_c in centers:
    plt.scatter(x_c, y_c, s=120, edgecolors='red', facecolors='none')

## Detect fixed marker

In [ ]:
mask_fix = make_mask(frame0, roi_fixed)
masked_fix = cv2.bitwise_and(frame0, frame0, mask=mask_fix)
binary_fix = color_filter(masked_fix, np.array([0, 20, 100]), np.array([255, 100, 255]), in_bgr=True)
fixed_centers = find_centers(binary_fix, min_area=100)
print('Fixed centers:', fixed_centers)
show_bgr(frame0)
for x_c, y_c in fixed_centers:
    plt.scatter(x_c, y_c, s=120, edgecolors='lime', facecolors='none')

## Calibration points

In [ ]:
mask_cal = make_mask(frame0, roi_calibration)
masked_cal = cv2.bitwise_and(frame0, frame0, mask=mask_cal)
binary_cal = color_filter(masked_cal, np.array([0, 20, 0]), np.array([255, 100, 100]), in_bgr=True)
cal_centers = find_centers(binary_cal, min_area=100)
if len(cal_centers) != 4:
    raise RuntimeError('Expected 4 calibration points, found ' + str(len(cal_centers)))
calibration_pixels = np.array(cal_centers, dtype=np.float32)
calibration_coords = np.array([[122, 42], [0, 42], [0, 0], [122, 0]], dtype=np.float32)

for x_c, y_c in cal_centers:
    xc, yc = perspective_coords(x_c, y_c, calibration_pixels, calibration_coords)
    print(f'Calibration dot -> ({xc:.1f}, {yc:.1f})')

show_bgr(frame0)
for x_c, y_c in cal_centers:
    plt.scatter(x_c, y_c, s=120, edgecolors='yellow', facecolors='none')

## Track full video

In [ ]:
cap = cv2.VideoCapture(str(video_path))
fourcc = cv2.VideoWriter_fourcc('M', 'J', 'P', 'G')
size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
fps = cap.get(cv2.CAP_PROP_FPS) or 25
out = cv2.VideoWriter(str(annotated_video_path), fourcc, fps, size, isColor=True)

pixel_centers = []
for fr in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))):
    ok, frame = cap.read()
    if not ok:
        print(f'Stopped early at frame {fr}')
        break
    mask = make_mask(frame, roi_moving)
    masked = cv2.bitwise_and(frame, frame, mask=mask)
    binary = color_filter(masked, lower_marker, upper_marker)
    centers = find_centers(binary, min_area=150)
    if centers:
        x_c, y_c = centers[0]
        pixel_centers.append((x_c, y_c))
        cv2.circle(frame, (int(x_c), int(y_c)), 10, (0, 0, 255), 2)
    out.write(frame)

cap.release()
out.release()
print(f'Saved annotated video to {annotated_video_path}')
print(f'Tracked {len(pixel_centers)} points')

## Save pixel and calibrated coordinates

In [ ]:
pixels_output.write_text('
'.join(f
 for x, y in pixel_centers))
calibrated = [perspective_coords(x, y, calibration_pixels, calibration_coords) for x, y in pixel_centers]
calibrated_output.write_text('
'.join(f
 for x, y in calibrated))

print(f'Pixel coords -> {pixels_output}')
print(f'Calibrated coords -> {calibrated_output}')
plt.plot([c[0] for c in calibrated], [c[1] for c in calibrated], 'k.', markersize=2)
plt.xlabel('x (cm)')
plt.ylabel('y (cm)')
plt.title('Tracked positions (calibrated)')
plt.grid(True)